# Barrido del umbral de decisión τ — Trade-off P_FA / P_MD

**Subsección 4.1** de la memoria (previa al barrido de tasa R=k/n).

**Justificación del diseño experimental:**
La detección de actividad en el Híbrido la realiza el `SupervisedAE` sobre la señal recibida `y`
(2n entradas reales), sin dependencia explícita de k. El modelo fue entrenado simultáneamente sobre
todos los k válidos a ρ fijo (modo multi-k), por lo que k=50 (R=0.5, punto central del barrido
principal) es el punto representativo. Fijar k rompe el barrido bidimensional (k, τ) sin pérdida
de generalidad para el análisis del umbral. Se evalúan solo P_FA y P_MD porque P_IE mezcla
detección con calidad de decodificación, que no es relevante aquí.

**Eficiencia:** p_active se recopila en un único pase por la red para cada (sistema, ρ);
τ se barre analíticamente sobre ese pool sin re-simular el canal.

## 0. Entorno Colab — Drive, repo y dependencias

Ejecuta estas tres celdas en orden la primera vez. En sesiones posteriores
basta con ejecutar la celda de `git pull` para tener el código actualizado.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_PATH = '/content/drive/MyDrive/tfg-datos'
if not os.path.isdir(REPO_PATH):
    !git clone https://github.com/monica793/tfg-datos.git {REPO_PATH}
else:
    !git -C {REPO_PATH} fetch origin
    !git -C {REPO_PATH} pull origin main
    !git -C {REPO_PATH} log -1 --oneline
print('Repo listo:', REPO_PATH)

In [ ]:
# NumPy estable antes/despues de Sionna (evita ImportError en numpy._core.umath)
%pip install -q -U 'numpy>=2.0,<2.3'
%pip install -q sionna
%pip install -q -U 'numpy>=2.0,<2.3'
# Si falla algun import tras el install, usa: Entorno de ejecucion -> Reiniciar sesion

In [ ]:
import os, sys

REPO_ROOT = '/content/drive/MyDrive/tfg-datos'
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print('cwd:', os.getcwd())

## 1. Barrido de umbral — ejecucion completa

La funcion `plot_tau_sweep()` hace todo:
1. Carga (o entrena si no existe checkpoint) los modelos Hibrido y E2E.
2. Recopila p_active en un unico pase por la red para cada (sistema, rho).
3. Barre tau analitico sobre el pool de muestras.
4. Genera la figura de publicacion y guarda PDF + PNG en `results/figures/`.
5. Guarda un CSV con los valores numericos en `results/tables/tau_sweep.csv`.

Para una ejecucion rapida de prueba reduce `n_batches=10`.

In [ ]:
import os, sys

REPO_ROOT = '/content/drive/MyDrive/tfg-datos'
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from evaluation.chapter4.s4_1_tau_sweep import plot_tau_sweep

results = plot_tau_sweep(
    n          = 100,
    k          = 50,
    p_empty    = 0.30,
    rho_dbs    = (0.0, 3.0),
    n_batches  = 100,      # use 10-20 for a quick smoke test
    batch_size = 5000,
    show       = True,
    alpha_mix  = 0.0,
)

## 2. Inspeccion rapida de resultados (opcional)

La celda siguiente imprime una tabla con los valores numericos clave:
el valor de tau donde P_FA ≈ P_MD (cruce) para cada sistema y SNR.

In [ ]:
import numpy as np
from evaluation.chapter4.s4_1_tau_sweep import THRESHOLDS_DEFAULT as TAUS

print(f'{"rho_db":>8}  {"sistema":>8}  {"tau_cross":>10}  '
      f'{"P_FA_cross":>12}  {"P_MD_cross":>12}')
print('-' * 60)

for (rho_db, sistema), (pfas, pmds) in results.items():
    # Encuentra el tau donde |P_FA - P_MD| es minimo (cruce aproximado)
    diff = np.abs(np.where(np.isnan(pfas) | np.isnan(pmds),
                           np.inf,
                           pfas - pmds))
    idx  = int(np.argmin(diff))
    print(f'{rho_db:>8.1f}  {sistema:>8}  {TAUS[idx]:>10.3f}  '
          f'{pfas[idx]:>12.4e}  {pmds[idx]:>12.4e}')